# Discovering Failure Patterns

In this notebook, you'll learn how to systematically review agent traces to discover failure patterns *before* building evaluators. This is a critical skill — if you don't know what's going wrong, you can't measure it, and you certainly can't fix it.

We'll work through a hands-on workflow:
1. **Read traces** from a restaurant booking agent and write notes about what went wrong
2. **Group similar problems** together with LLM assistance
3. **Prioritize** which problems to tackle first
4. **Fix the top problem** with a simple prompt change
5. **Bridge into evaluator design** by turning a problem category into a binary pass/fail check

By the end, you'll see that understanding your failures first makes everything else — prompt fixes, evaluator design, regression testing — dramatically easier.

## About the Data

The traces we'll review come from a **restaurant booking agent** that handles reservations, cancellations, and booking lookups. These are the same traces used in Module 03 (Agentic Metrics), so you'll see a deliberate narrative thread across the workshop.

Each trace is a single conversation between a user and the agent, stored as a JSON object with a `conversation_id` and an ordered list of messages (each with a `role` and `content` field).

> **📋 A note on trace sampling in production**
>
> In a real production system, you'd sample traces from an observability platform like **Amazon CloudWatch** or **LangSmith** rather than loading them all from a file. You'd filter by time window, error rates, or user-reported issues to find the most informative conversations to review.
>
> For this workshop, we use all 20 traces for simplicity. In production, aim to review **at least 100 traces** to get a representative picture of your agent's failure modes. The more traces you review, the more confident you can be that your problem categories cover the real issues.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
import json
from collections import Counter
from IPython.display import display, Markdown, HTML

MODEL_ID = "us.anthropic.claude-sonnet-4-6"
bedrock_model = BedrockModel(model_id=MODEL_ID)

In [ ]:
with open("data/raw_traces.json") as f:
    raw_traces = json.load(f)

print(f"Loaded {len(raw_traces)} raw traces")

In [ ]:
# Let's look at the first trace to understand the data format
sample_trace = raw_traces[0]

print(f"Conversation ID: {sample_trace['conversation_id']}")
print(f"Number of messages: {len(sample_trace['messages'])}")
print("---")

for i, msg in enumerate(sample_trace["messages"], 1):
    role = msg.get("role", "unknown")
    content = msg.get("content", "")
    print(f"\nTurn {i} — {role}:")
    print(f"  {content}")

## ⚠️ Common Bedrock Errors and How to Fix Them

If you run into issues when running the code cells above, here are the most common causes and fixes:

| Problem | What you'll see | How to fix it |
|---------|----------------|---------------|
| **Model not enabled** | `AccessDeniedException` or "model is not enabled" | Go to the [Amazon Bedrock console](https://console.aws.amazon.com/bedrock/) → **Model access** → Enable **Claude** models |
| **Wrong region** | `Could not connect to the endpoint URL` | Make sure your AWS region has Claude available. Check your `AWS_DEFAULT_REGION` environment variable or boto3 config |
| **IAM permissions** | `AccessDeniedException` mentioning IAM | Verify your IAM role or user has the `bedrock:InvokeModel` permission |
| **Throttling** | `ThrottlingException` or rate limit errors | Add `import time` and `time.sleep(2)` between calls if you see throttling errors. This is especially common when re-running multiple traces back-to-back |

## Section 2: Reviewing Traces and Writing Notes

Now that the data is loaded, it's time to read through the traces one at a time and write down what you notice. This is the most important step in the whole module — everything that follows (grouping, prioritizing, fixing) depends on what you observe here.

We'll display each trace with a note-taking cell right next to it, so you can review and write without scrolling back and forth.

In [ ]:
def display_trace(trace, trace_number=None):
    """Display a single trace with numbered turns, role labels, and highlighted tool calls.

    Args:
        trace: A dict with 'conversation_id' and 'messages' list.
        trace_number: Optional display number (e.g., shows 'Trace 3' in the header).
    """
    conversation_id = trace.get("conversation_id", "(unknown)")
    messages = trace.get("messages", [])

    lines = []

    # Header
    if trace_number is not None:
        lines.append(f"━━━ Trace {trace_number} ━━━")
    lines.append(f"**Conversation ID:** `{conversation_id}`")
    lines.append("")

    # Handle empty messages
    if not messages:
        lines.append("(no messages in this trace)")
        display(Markdown("\n".join(lines)))
        return

    # Display each message as a numbered turn
    for i, msg in enumerate(messages, 1):
        role = msg.get("role", None)
        content = msg.get("content", None)

        # Determine the label for this turn
        if content is not None and content.startswith("TOOL_CALL["):
            label = "⚠️ Tool Result"
        elif role == "user":
            label = "👤 User"
        elif role == "agent":
            label = "🤖 Agent"
        elif role is None:
            label = "(missing role)"
        else:
            label = f"({role})"

        # Handle missing content
        display_content = content if content is not None else "(missing content)"

        lines.append(f"**Turn {i} — {label}**")
        lines.append(f"> {display_content}")
        lines.append("")

    display(Markdown("\n".join(lines)))

In [ ]:
import sys, io

def suggest_note(trace):
    """Ask the LLM to suggest a note about what went wrong in a trace.
    
    This gives you a starting point — read the trace yourself first,
    then accept the suggestion or write your own version.
    
    Args:
        trace: A dict with 'conversation_id' and 'messages' list.
    
    Returns:
        A string containing the LLM's suggested note.
    """
    note_agent = Agent(
        model=bedrock_model,
        system_prompt=(
            "You are helping label AI agent traces. "
            "Identify the first thing that went wrong — it could be a tool error, "
            "bad reasoning by the agent, a misunderstanding, or anything else. "
            "Write ONE or TWO sentences, max 50 words. Be specific and concise about what happened."
        )
    )
    
    # Format the trace as text for the LLM
    trace_text = f"Conversation ID: {trace['conversation_id']}\n\n"
    for i, msg in enumerate(trace.get("messages", []), 1):
        role = msg.get("role", "unknown")
        content = msg.get("content", "")
        trace_text += f"Turn {i} — {role}: {content}\n"
    
    prompt = f"""What is the first thing that went wrong in this trace?

{trace_text}"""
    
    # Suppress streaming output so the LLM's token-by-token generation
    # doesn't clutter the notebook. Only the final result is displayed.
    _stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        response = note_agent(prompt)
    finally:
        sys.stdout = _stdout
    
    return str(response).strip()

### 💡 Focus on the first failure in each trace

When you're reading through a trace, look for the **first thing that goes wrong**. Once an early step fails, everything after it is often a consequence of that initial problem.

For example, if a tool call returns an error in Turn 3 and the agent gives a wrong answer in Turn 5, the root cause is usually the tool failure — not the agent's response. Fixing the upstream problem (how the agent handles tool errors) often fixes the downstream symptoms automatically.

Write your notes about the **first failure**, not the last one.

> **🚨 Why you should read traces before building evaluators**
>
> Imagine you build an evaluator that checks *"Did the agent provide accurate booking details?"* before you've read the traces. You might spend days calibrating that evaluator — tuning the prompt, testing edge cases, adjusting thresholds — only to discover that the real problem is the agent **confirming actions that actually failed**. That's a completely different issue, and your carefully built evaluator wouldn't catch it.
>
> This is what we call building evaluators backwards. It's tempting because it feels productive, but it wastes time measuring the wrong things. Read first, then build.

### Trace Review

Below, each trace is followed by a note-taking cell. The first 3 traces have example notes filled in to show the level of detail we're looking for. For the remaining 7, an LLM will suggest a note — read the trace yourself first, then accept the suggestion or write your own corrected version.

In [ ]:
display_trace(raw_traces[0], trace_number=1)

In [ ]:
# Example note — here's the level of detail to aim for.
note_1 = {
    "conversation_id": "393b219a-ff56-4edb-a494-ab12c3fa1248",
    "note": "Agent confirmed reservation but the GetBookingDetails tool returned an error — agent acted as if the booking went through when it had no connection to the database"
}

In [ ]:
display_trace(raw_traces[1], trace_number=2)

In [ ]:
# Example note — here's the level of detail to aim for.
note_2 = {
    "conversation_id": "0f186b8e-bf58-44ab-a117-a57ed6141c80",
    "note": "Agent said cancellation was processed successfully, but the GetCustomerProfile tool failed with an authentication error — the agent never actually verified the reservation existed before claiming it was cancelled"
}

In [ ]:
display_trace(raw_traces[2], trace_number=3)

In [ ]:
# Example note — here's the level of detail to aim for.
note_3 = {
    "conversation_id": "faf64bb8-4a60-4946-adf8-bd50468bc3b5",
    "note": "GenBookingArgs tool returned an error about multiple Friday reservations, but the agent ignored this and told the user the cancellation was successful — also fabricated refund details without any data from a tool"
}

In [ ]:
display_trace(raw_traces[3], trace_number=4)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_4 = suggest_note(raw_traces[3])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_4 with your note in quotes.
note_4 = {
    "conversation_id": raw_traces[3]["conversation_id"],
    "note": _suggestion_4  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_4}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
display_trace(raw_traces[4], trace_number=5)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_5 = suggest_note(raw_traces[4])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_5 with your note in quotes.
note_5 = {
    "conversation_id": raw_traces[4]["conversation_id"],
    "note": _suggestion_5  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_5}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
display_trace(raw_traces[5], trace_number=6)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_6 = suggest_note(raw_traces[5])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_6 with your note in quotes.
note_6 = {
    "conversation_id": raw_traces[5]["conversation_id"],
    "note": _suggestion_6  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_6}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
display_trace(raw_traces[6], trace_number=7)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_7 = suggest_note(raw_traces[6])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_7 with your note in quotes.
note_7 = {
    "conversation_id": raw_traces[6]["conversation_id"],
    "note": _suggestion_7  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_7}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
display_trace(raw_traces[7], trace_number=8)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_8 = suggest_note(raw_traces[7])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_8 with your note in quotes.
note_8 = {
    "conversation_id": raw_traces[7]["conversation_id"],
    "note": _suggestion_8  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_8}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
display_trace(raw_traces[8], trace_number=9)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_9 = suggest_note(raw_traces[8])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_9 with your note in quotes.
note_9 = {
    "conversation_id": raw_traces[8]["conversation_id"],
    "note": _suggestion_9  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_9}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
display_trace(raw_traces[9], trace_number=10)

In [ ]:
# LLM suggests a note — read the trace above, then accept or write your own.
_suggestion_10 = suggest_note(raw_traces[9])

# To accept the LLM's suggestion, just run this cell as-is.
# To write your own, replace _suggestion_10 with your note in quotes.
note_10 = {
    "conversation_id": raw_traces[9]["conversation_id"],
    "note": _suggestion_10  # ← Replace with your own note in quotes if the LLM got it wrong
}

print(f"🤖 LLM suggestion: {_suggestion_10}")
print("   Accept this by doing nothing, or try your hand at writing your own label above.")

In [ ]:
# Combine all notes for use in the next section
all_notes = [note_1, note_2, note_3, note_4, note_5, note_6, note_7, note_8, note_9, note_10]
print(f"Total notes collected: {len(all_notes)}")
print()
for i, note in enumerate(all_notes, 1):
    print(f"{i}. [{note['conversation_id'][:12]}...] {note['note']}")

## Section 3: Grouping Similar Problems Together

You've read through the traces and written notes about what went wrong. Now it's time to organize those notes into **actionable categories** — groups of similar problems that you can prioritize and fix.

This is where individual observations turn into a clear picture of your agent's failure patterns. Instead of 10 separate notes, you'll end up with 3-6 distinct problem categories that tell you exactly what's going wrong and how often.

In [ ]:
print(f"Reviewing {len(all_notes)} notes:\n")
for i, note in enumerate(all_notes, 1):
    print(f"{i}. [{note['conversation_id'][:12]}...] {note['note']}")

### 💡 The LLM assists with grouping, but you must read the traces yourself first

In the next cell, we'll use an LLM to suggest how to group your notes into problem categories. This is a useful shortcut — it's faster than sorting through notes manually.

But here's the important part: **the LLM hasn't seen the traces**. It's working entirely from your notes. If your notes are vague ("something went wrong"), the groupings will be vague too. If your notes are specific ("agent confirmed cancellation but the DeleteBooking tool returned an error"), the groupings will be specific and actionable.

This is why reading the traces yourself matters. The LLM can organize your observations, but it can't make observations for you.

In [ ]:
grouping_agent = Agent(
    model=bedrock_model,
    system_prompt="You are helping a workshop participant analyze notes about an AI agent's failures. Be specific and actionable in your suggestions."
)

GROUPING_PROMPT = """Here are notes from reviewing {count} conversations with a restaurant booking agent. 
Each note describes something that went wrong or seemed concerning.

Notes:
{formatted_notes}

Based on these notes, suggest 3-6 distinct problem categories. For each category provide:
1. A specific, actionable name (e.g., "Agent confirms action succeeded when the tool call failed" — NOT vague names like "bad response")
2. A one-sentence definition
3. Which conversation IDs belong to this category

Format your response as a numbered list."""

In [ ]:
formatted_notes = "\n".join(
    f"- [{n['conversation_id'][:12]}...] {n['note']}" for n in all_notes
)

response = grouping_agent(GROUPING_PROMPT.format(count=len(all_notes), formatted_notes=formatted_notes))
display(Markdown(str(response)))

In [ ]:
# Parse the LLM's grouping into a structured dict so you can use it directly if you want.
# This calls the LLM again to extract structured data from its own response.

import re

def _parse_dict_from_llm(text):
    """Extract a dict of string keys to string/list values from LLM output using regex.
    
    Handles two patterns:
    - {"key": "value", ...} for simple string-to-string dicts
    - {"key": ["val1", "val2"], ...} for string-to-list-of-strings dicts
    """
    text = str(text)
    # Strip markdown code fences if present
    fence_match = re.search(r'```(?:python)?\s*\n?(.*?)\n?```', text, re.DOTALL)
    if fence_match:
        text = fence_match.group(1)
    
    result = {}
    # Pattern: "key": ["val1", "val2", ...]
    list_pattern = re.findall(r'["\']([^"\']+)["\']\s*:\s*\[([^\]]*)\]', text)
    if list_pattern:
        for key, values_str in list_pattern:
            values = re.findall(r'["\']([^"\']+)["\']', values_str)
            result[key] = values
        return result
    
    # Pattern: "key": "value"
    str_pattern = re.findall(r'["\']([^"\']+)["\']\s*:\s*["\']([^"\']+)["\']', text)
    if str_pattern:
        for key, value in str_pattern:
            result[key] = value
        return result
    
    return result

parse_agent = Agent(
    model=bedrock_model,
    system_prompt="You extract structured data from text. Return only valid Python code, no explanation."
)

_parse_prompt = f"""Convert the following problem categories into a Python dict.
The dict should map category names (strings) to lists of conversation ID prefixes (strings).
Return ONLY the dict literal, nothing else.

{str(response)}"""

_stdout = sys.stdout
sys.stdout = io.StringIO()
try:
    _parsed = parse_agent(_parse_prompt)
finally:
    sys.stdout = _stdout

generated_problem_categories = _parse_dict_from_llm(str(_parsed))
if generated_problem_categories:
    print(f"Generated {len(generated_problem_categories)} categories from LLM grouping:")
    for cat, ids in generated_problem_categories.items():
        print(f"  {len(ids)}x  {cat}")
else:
    print("Could not parse LLM response into a dict.")
    print("You'll need to create problem_categories manually in the next cell.")

### What makes a good problem category name?

The names you give your categories matter. A good name tells you exactly what to look for and what to fix. A bad name is too vague to act on.

**Good category names** (specific, actionable):
- ✅ *"Agent confirms action succeeded when the tool call failed"* — You know exactly what's happening and where to look
- ✅ *"Agent fabricates details not present in any tool response"* — Clear what to look for in the trace

**Bad category names** (too vague to act on):
- ❌ *"Bad response"* — What kind of bad? Wrong information? Wrong tone? Wrong format?
- ❌ *"Error handling"* — Which errors? What handling? This could mean a dozen different things

When you finalize your categories below, aim for names that someone unfamiliar with the traces could read and immediately understand what the problem is.

In [ ]:
# Review the LLM's suggestions above, then create your final categories.
# Merge, split, or rename categories as needed. The LLM's output is a starting point, not the answer.

problem_categories = {
    "Agent confirms action succeeded when the tool call failed": [
        # List conversation IDs that belong to this category
        "393b219a-ff56-4edb-a494-ab12c3fa1248",
        # Add more...
    ],
    # Add more categories...
}

######################## SHORTCUT ########################
# uncomment the next line to use the LLM-generated categories directly.
#problem_categories = generated_problem_categories

In [ ]:
print("Problem Category Frequencies:\n")
for category, conv_ids in sorted(problem_categories.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {len(conv_ids)}x  {category}")
print(f"\nTotal categories: {len(problem_categories)}")

## Section 4: Prioritizing Problems

You now have a set of problem categories with frequency counts. But not all problems are equally important — a problem that happens often *and* misleads users is more urgent than a rare cosmetic issue.

In this section, you'll rank your problem categories by combining **how often** they occur (frequency) with **how bad** they are (severity). This gives you a clear priority list so you know exactly where to focus your effort first.

In [ ]:
print("Problem Category Frequencies:\n")
print(f"{' Category':<60} {'Count':>5}")
print("-" * 67)
for category, conv_ids in sorted(problem_categories.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{category:<60} {len(conv_ids):>5}")

### Assigning Severity Ratings

Frequency alone doesn't tell the whole story. A problem that happens in 2 out of 10 traces but **misleads the user** is more urgent than one that happens in 5 traces but is purely cosmetic.

Assign one of these severity levels to each of your problem categories:

| Severity | Weight | Meaning |
|----------|--------|---------|
| **Critical** | 4 | User is misled or harmed (e.g., agent confirms a booking that never happened) |
| **High** | 3 | Task fails silently (e.g., agent doesn't acknowledge a tool error) |
| **Medium** | 2 | Poor experience but task completes (e.g., agent asks for info it already has) |
| **Low** | 1 | Cosmetic (e.g., awkward phrasing, unnecessary apology) |

### 💡 LLM-suggested severity ratings are a starting point

The LLM will suggest a severity rating for each category, but these are starting points — not final answers. You should review each rating and adjust based on your domain knowledge. A problem that seems "High" to the LLM might be "Critical" in your business context if it affects revenue or user trust.

In [ ]:
# Ask the LLM to suggest severity ratings for each problem category.

severity_agent = Agent(
    model=bedrock_model,
    system_prompt=(
        "You assign severity ratings to problem categories for an AI agent. "
        "Use exactly one of: Critical, High, Medium, Low. "
        "Critical = user is misled or harmed. High = task fails silently. "
        "Medium = poor experience but task completes. Low = cosmetic. "
        "Return ONLY a valid Python dict mapping category name to severity string."
    )
)

_categories_list = "\n".join(f"- {cat}" for cat in problem_categories.keys())
_severity_prompt = f"""Assign a severity rating (Critical, High, Medium, or Low) to each of these problem categories.
Return a Python dict mapping each category name to its rating.

Categories:
{_categories_list}"""

_stdout = sys.stdout
sys.stdout = io.StringIO()
try:
    _severity_response = severity_agent(_severity_prompt)
finally:
    sys.stdout = _stdout

try:
    generated_severity_ratings = _parse_dict_from_llm(str(_severity_response))
    if generated_severity_ratings:
        print("Generated severity ratings:")
        for cat, sev in generated_severity_ratings.items():
            print(f"  {sev:<10} {cat}")
    else:
        print("Could not parse LLM response.")
        print("You'll need to assign severity_ratings manually in the next cell.")
        generated_severity_ratings = {}
except Exception as e:
    print(f"Could not parse LLM response: {e}")
    print("You'll need to assign severity_ratings manually in the next cell.")
    generated_severity_ratings = {}

In [ ]:
SEVERITY_WEIGHTS = {
    "Critical": 4,  # User is misled or harmed
    "High": 3,      # Task fails silently
    "Medium": 2,    # Poor experience but task completes
    "Low": 1        # Cosmetic
}

# Assign a severity rating to each of your problem categories.
# Replace the ratings below with your own assessment.

severity_ratings = {
    "Agent confirms action succeeded when the tool call failed": "Critical",
    # Add your other categories here...
}

######################## SHORTCUT ########################
# uncomment the next line to use the LLM-generated severity ratings directly.
#severity_ratings = generated_severity_ratings

In [ ]:
def compute_priorities(problem_categories, severity_ratings):
    """Compute priority scores as frequency × severity weight.

    Args:
        problem_categories: Dict mapping category name to list of conversation IDs.
        severity_ratings: Dict mapping category name to severity level string.

    Returns:
        List of dicts with 'category', 'frequency', 'severity', 'weight', 'priority_score',
        sorted by priority_score descending.

    Raises:
        ValueError: If keys in problem_categories and severity_ratings don't match.
    """
    if not problem_categories:
        return []

    cat_keys = set(problem_categories.keys())
    sev_keys = set(severity_ratings.keys())
    if cat_keys != sev_keys:
        missing_in_severity = cat_keys - sev_keys
        extra_in_severity = sev_keys - cat_keys
        parts = []
        if missing_in_severity:
            parts.append(f"Missing severity ratings for: {missing_in_severity}")
        if extra_in_severity:
            parts.append(f"Extra severity ratings for: {extra_in_severity}")
        raise ValueError("; ".join(parts))

    results = []
    for category, conv_ids in problem_categories.items():
        severity = severity_ratings[category]
        weight = SEVERITY_WEIGHTS[severity]
        frequency = len(conv_ids)
        results.append({
            "category": category,
            "frequency": frequency,
            "severity": severity,
            "weight": weight,
            "priority_score": frequency * weight,
        })

    results.sort(key=lambda x: x["priority_score"], reverse=True)
    return results

In [ ]:
priorities = compute_priorities(problem_categories, severity_ratings)

print("Priority Ranking:\n")
MAX_CAT = 85
print(f"{'#':<3} {'Category':<{MAX_CAT}} {'Freq':>4} {'Severity':<9} {'Score':>5}")
print("-" * (MAX_CAT + 25))
for i, p in enumerate(priorities, 1):
    cat = p['category'][:MAX_CAT-1] + '…' if len(p['category']) > MAX_CAT else p['category']
    print(f"{i:<3} {cat:<{MAX_CAT}} {p['frequency']:>4} {p['severity']:<9} {p['priority_score']:>5}")

### What to do with each problem

Now that you have a ranked list, use this decision flowchart to decide what action to take for each problem category:

```
Problem discovered
  └─ Can I fix it by changing the prompt?
       ├─ YES → Fix it now. Add a regression test. Done.
       └─ NO → Is it frequent enough to justify building a checker?
              ├─ YES → Can I catch it with a simple code check (string match, regex)?
              │    ├─ YES → Write the check. Cheap and fast.
              │    └─ NO → Build an LLM-as-Judge evaluator (see Module 02).
              └─ NO → Log it. Revisit if frequency increases.
```

The key insight: **start with the cheapest fix**. A 5-minute prompt edit that eliminates a problem is always better than a multi-day evaluator build. Save the heavy machinery (LLM-as-Judge) for problems that can't be fixed at the source.

In the next section, we'll apply this flowchart to the top-priority problem and fix it with a prompt change.

## Section 5: Fix the Top Problem

Your priority list almost certainly has **"Agent confirms action succeeded when the tool call failed"** at or near the top. This is the "hallucinated success" pattern — the agent tells the user everything worked when it didn't.

Following the decision flowchart: *Can we fix it by changing the prompt?* Let's find out. We'll look at the agent's system prompt, identify the gap that causes this behavior, write an improved version, and re-run traces to see if the fix works.

In [ ]:
ORIGINAL_SYSTEM_PROMPT = """You are a helpful restaurant booking assistant. You help customers with:
- Making new reservations
- Cancelling existing reservations  
- Looking up booking details
- Modifying reservations

When helping customers, be polite and professional. Collect all necessary information 
including the customer's name, party size, preferred date and time, and any special requests.

If you need to use tools to look up or modify bookings, do so proactively. 
Always confirm the details with the customer before finalizing any changes.

Available tools:
- GetBookingDetails: Look up reservation information
- CreateBooking: Make a new reservation
- DeleteBooking: Cancel an existing reservation
- GetCustomerProfile: Retrieve customer information
- GenBookingArgs: Generate booking parameters from conversation context"""

print("Original system prompt:")
print(ORIGINAL_SYSTEM_PROMPT)

### Spot the gap

Read the prompt above carefully. It tells the agent to be polite, collect information, use tools proactively, and confirm details with the customer.

But what happens when a tool call **fails**? The prompt says nothing about it. There's no instruction to report errors honestly, no guidance on what to do when an action can't be completed.

So the agent does what LLMs do by default — it continues as if everything worked. It "confirms" a booking that was never made, "cancels" a reservation that couldn't be found, and fabricates details to fill the gaps. The prompt never told it to do otherwise.

### Write an improved prompt

The fix is straightforward: add explicit instructions about what the agent should do when a tool call fails. For example:

- If a tool call returns an error, inform the customer that the action could not be completed
- Explain what happened (in plain language, not technical error codes)
- Never confirm an action that failed
- Suggest alternatives (try again later, contact the restaurant directly)

Try writing your own improved version in the cell below.

In [ ]:
# Write your improved system prompt below.
# Hint: add instructions about what the agent should do when a tool call fails.
improved_prompt = ORIGINAL_SYSTEM_PROMPT + """

If a tool call returns an error, inform the customer that the action could not be completed 
and explain what happened. Never confirm an action that failed. If you cannot complete a 
request due to a tool error, apologize and suggest the customer try again later or contact 
the restaurant directly."""

print("Improved system prompt (showing the addition):")
print(improved_prompt[len(ORIGINAL_SYSTEM_PROMPT):])

### How we test the prompt fix: persona simulation

To see whether the improved prompt actually changes the agent's behavior, we replay the original conversation against the improved agent. But we can't just send all the user messages at once — the failure happens mid-conversation, and we need to see how the agent responds at each turn.

The approach: a **simulated user** (another LLM) plays the role of the original customer. It follows the original script when the agent behaves as expected, but responds naturally if the agent diverges — for example, if the improved agent reports an error instead of confirming success, the simulated user might ask "oh, what happened?" rather than continuing with the next scripted message.

Tool results from the original trace are injected at the same points, so both the original and improved agents face the same environment. The only difference is the system prompt.

> For a deeper treatment of persona simulation — including calibrating personas, handling diverse user behaviors, and running simulations at scale — see the persona simulation module elsewhere in this workshop.

In [ ]:
def create_simulated_user(original_trace):
    """Create a simulated user agent that follows the original trace's script.
    
    The simulated user sends the same messages as the original customer.
    If the agent diverges from the original (e.g., reports an error instead of
    confirming success), the simulated user responds naturally.
    
    Args:
        original_trace: The original trace dict with 'messages' list.
    
    Returns:
        A Strands Agent configured as the simulated user.
    """
    # Format the original trace for the persona's context
    trace_script = "\n".join(
        f"Turn {i} — {m.get('role', '?')}: {m.get('content', '')}"
        for i, m in enumerate(original_trace.get('messages', []), 1)
    )
    
    persona_prompt = f"""You are simulating a customer in a restaurant booking conversation.
Here is how the original conversation went:

{trace_script}

Rules:
- Follow the customer's messages from the original trace as closely as possible.
- If the agent responds similarly to the original, send the next customer message from the script.
- If the agent says something DIFFERENT from the original (e.g., reports an error, asks a new question), respond naturally as the customer would.
- Keep responses short and natural (1-2 sentences).
- When the conversation reaches a natural end, output exactly: [END]
- If you've sent all the customer messages from the script and the conversation feels complete, output: [END]"""
    
    return Agent(model=bedrock_model, system_prompt=persona_prompt)


def replay_conversation(booking_agent, original_trace, max_turns=20):
    """Replay a trace using a persona simulator against the booking agent.
    
    The simulated user follows the original trace's script. Tool results from
    the original trace are injected at the appropriate points so both agents
    face the same environment.
    
    Args:
        booking_agent: The Strands Agent to test (with original or improved prompt).
        original_trace: The original trace dict.
        max_turns: Maximum number of turns before stopping.
    
    Returns:
        A list of dicts with 'role' and 'content' representing the replayed conversation.
    """
    sim_user = create_simulated_user(original_trace)
    
    # Extract tool results from the original trace (in order)
    tool_results = [
        m['content'] for m in original_trace.get('messages', [])
        if m.get('content', '').startswith('TOOL_CALL[')
    ]
    tool_idx = 0
    
    # Get the first user message to kick things off
    first_user_msgs = [
        m['content'] for m in original_trace.get('messages', [])
        if m.get('role') == 'user'
    ]
    if not first_user_msgs:
        return []
    
    conversation = []
    
    # Suppress streaming output for both agents
    _stdout = sys.stdout
    
    # Start with the first user message
    user_msg = first_user_msgs[0]
    conversation.append({'role': 'user', 'content': user_msg})
    
    for turn in range(max_turns):
        # Booking agent responds to the user message
        # If there's a tool result to inject, include it in the prompt
        agent_input = user_msg
        if tool_idx < len(tool_results):
        # Inject the tool result from the original trace so the agent faces
        # the same environment as the original. This isolates the effect of
        # the prompt change — same tool failures, different agent behavior.
            agent_input += f"\n\n[System: Tool result: {tool_results[tool_idx]}]"
            tool_idx += 1
            conversation.append({'role': 'tool', 'content': tool_results[tool_idx - 1]})
        
        sys.stdout = io.StringIO()
        try:
            agent_response = booking_agent(agent_input)
        finally:
            sys.stdout = _stdout
        
        agent_text = str(agent_response).strip()
        conversation.append({'role': 'agent', 'content': agent_text})
        
        # Simulated user responds
        sys.stdout = io.StringIO()
        try:
            user_response = sim_user(f"The agent said: {agent_text}")
        finally:
            sys.stdout = _stdout
        
        user_text = str(user_response).strip()
        
        # Check if conversation is over
        if '[END]' in user_text:
            break
        
        conversation.append({'role': 'user', 'content': user_text})
        user_msg = user_text
    
    return conversation

In [ ]:
def display_comparison(original_trace, replayed_conversation):
    """Display side-by-side comparison of original trace vs replayed conversation."""
    trace_id = original_trace.get('conversation_id', 'unknown')
    lines = [f"### Trace: `{trace_id}`\n"]
    
    # Original trace
    lines.append("**Original conversation (what actually happened):**\n")
    for msg in original_trace.get('messages', []):
        content = msg.get('content', '')
        role = msg.get('role', 'unknown')
        if content.startswith('TOOL_CALL['):
            lines.append(f"> ⚠️ {content}\n")
        else:
            emoji = '👤' if role == 'user' else '🤖'
            lines.append(f"> {emoji} {content}\n")
    
    # Replayed conversation
    lines.append("\n**Replayed with improved prompt:**\n")
    for msg in replayed_conversation:
        role = msg.get('role', 'unknown')
        content = msg.get('content', '')
        if role == 'tool':
            lines.append(f"> ⚠️ {content}\n")
        elif role == 'user':
            lines.append(f"> 👤 {content}\n")
        else:
            lines.append(f"> 🤖 {content}\n")
    
    lines.append("---\n")
    display(Markdown("\n".join(lines)))

In [ ]:
# Configure how many traces to replay (start with 1, increase once you're comfortable)
TRACES_TO_REPLAY = 1

# Find traces that contain tool errors (good candidates for seeing the prompt fix in action)
tool_error_traces = [
    t for t in raw_traces 
    if any("TOOL_CALL[" in msg.get("content", "") and "Error" in msg.get("content", "")
           for msg in t.get("messages", []))
]

print(f"Found {len(tool_error_traces)} traces with tool errors.")
print(f"Replaying {TRACES_TO_REPLAY} trace(s) with the improved prompt...\n")

for trace in tool_error_traces[:TRACES_TO_REPLAY]:
    # Create a fresh improved agent for this replay
    replay_agent = Agent(model=bedrock_model, system_prompt=improved_prompt)
    
    # Replay the conversation with persona simulation
    replayed = replay_conversation(replay_agent, trace)
    
    # Show original vs replayed side by side
    display_comparison(trace, replayed)

### 💡 The cheapest fix is often the best fix

This fix took about 5 minutes — we added a few sentences to the system prompt. Compare that to the cost of building an LLM-as-Judge evaluator for the same problem: writing the judge prompt, collecting test cases, calibrating the judge, running it in CI. That's days of work to *detect* a problem that we just *eliminated* with a prompt edit.

This doesn't mean evaluators are useless — far from it. But the decision flowchart matters: **fix what you can fix first**, then build evaluators for the problems that remain. You'll end up with fewer evaluators, each one focused on a problem that can't quickly be fixed, and may need more information to correct.

> **📋 Who owns the problem list?**
>
> In production, it helps to have one person who owns the problem list for each agent. This "benevolent dictator" reads the traces regularly, maintains the problem categories, and decides what to fix vs. what to evaluate. Shared ownership often means no one reads the traces — everyone assumes someone else is doing it.
>
> This doesn't have to be a full-time role. Even 30 minutes a week of trace review by one dedicated person catches more issues than an elaborate evaluation pipeline that no one monitors.

## Section 6: Bridging into Evaluator Design

Not every problem can be fixed with a prompt change. Some failure modes are subtle, context-dependent, or emerge from the interaction between the agent and its tools in ways that no single prompt instruction can prevent.

For those problems, you need **automated checking** — an evaluator that can review traces and flag failures without a human reading every conversation. This section shows how to take a discovered problem category and turn it into a binary pass/fail evaluator prompt, bridging the gap between what you learned here and the evaluator design workflow taught in Module 02.

### Pick a problem and write a binary question

Go back to your prioritized list and select one problem category — **not** the one you already fixed with a prompt change. This should be a problem that can't easily be eliminated at the source.

Now write it as a **single binary yes/no question** that a judge could answer by reading a trace. The question should be specific enough that two people reading the same trace would give the same answer.

For example:
> *"Did the agent fabricate specific details like table numbers, refund amounts, or booking confirmations without data from a tool call?"*

Notice how this question is concrete and observable — you can answer it by checking whether the details the agent mentioned actually appeared in a tool response. Vague questions don't work because different readers will interpret them differently. Examples of questions that are too vague to be useful:
- *"Was the agent helpful?"* — helpful how? By what standard?
- *"Did the agent make a mistake?"* — what counts as a mistake?
- *"Was the response accurate?"* — accurate about what specifically?

In [ ]:
# Pick a problem category from your priority list (not the one you already fixed).
# Write it as a single yes/no question that a judge could answer by reading a trace.

binary_question = "Did the agent fabricate specific details (like table numbers, refund amounts, or booking confirmations) without data from a tool call?"

print(f"Binary question: {binary_question}")

In [ ]:
JUDGE_PROMPT_TEMPLATE = """You are evaluating a conversation between a user and a restaurant booking agent.

Review the conversation below and answer this question:
{binary_question}

Respond with a JSON object:
{{"verdict": "PASS" or "FAIL", "reasoning": "one sentence explanation"}}

Conversation:
{trace_text}"""

In [ ]:
judge_agent = Agent(
    model=bedrock_model,
    system_prompt="You are a precise evaluator. Answer exactly as instructed."
)

# Test against 2-3 traces
test_traces = raw_traces[:3]

for trace in test_traces:
    trace_text = "\n".join(
        f"{msg.get('role', 'unknown')}: {msg.get('content', '')}" 
        for msg in trace["messages"]
    )
    
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        binary_question=binary_question,
        trace_text=trace_text
    )
    
    response = judge_agent(prompt)
    print(f"\nTrace {trace['conversation_id'][:12]}...")
    print(f"Judge says: {response}")
    print("---")

This is just a sketch — a quick test to see if your binary question produces useful verdicts. For the full judge calibration workflow (testing against labeled examples, measuring agreement, tuning the prompt), see Module 02's [Evaluating your Judge](../02-quality-metrics/03_Evaluating_your_Judge.ipynb) notebook.

### 💡 One problem → one question → one evaluator

Each problem category should map to exactly one yes/no question. This is the bridge between discovering problems (what you did in this module) and building evaluators (what Module 02 teaches). One problem → one question → one evaluator.

If you find yourself writing a question that covers multiple failure modes, split it. A judge prompt that tries to check for three things at once will be unreliable — it's better to have three focused evaluators that each do one thing well.

## Section 7: Key Takeaways

Here's what to take away from this module:

- **Review before building evaluators** — You can't measure what you don't understand. Reading traces first tells you what to measure.
- **Fix problems before automating detection** — Many failures disappear with a simple prompt change. Save evaluator-building for problems that can't be fixed at the source.
- **LLMs assist but don't replace reading traces** — The LLM helped group your notes, but it couldn't have written them. Your observations are the foundation.
- **Revisit regularly** — Failure patterns change as you update prompts, switch models, or add features. Make trace review a recurring practice, not a one-time exercise.

### Connections to other modules

- **Module 01 — Operational Metrics** ([01-Operational-Metrics.ipynb](../01-operational-metrics/01-Operational-Metrics.ipynb)): Operational anomalies like high latency or unusual token counts are signals for where to start your trace review. If a set of conversations shows unusually high token usage, that's a good place to look first.
- **Module 02 — Quality Metrics** ([03_Evaluating_your_Judge.ipynb](../02-quality-metrics/03_Evaluating_your_Judge.ipynb)): The failure modes you discovered here become the criteria for LLM-as-Judge evaluators. The binary pass/fail pattern you sketched in Section 6 is taught in depth in Module 02's judge calibration notebook.
- **Module 04 — Agentic Metrics** ([01-Agentic-Metrics.ipynb](../04-agentic-metrics/01-Agentic-Metrics.ipynb)): These same restaurant booking traces are used for agentic evaluation in Module 04. The problem categories you built here directly inform what metrics to track there.

In production, repeat this review process after every significant change: new features, prompt updates, model switches, or tool changes. What fails today may not fail tomorrow, and new failures will emerge. A 30-minute weekly trace review catches more issues than any automated pipeline running unmonitored.